In [2]:
import langchain
import duckdb
import pandas as pd
from langchain_community.document_loaders import DataFrameLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
model = "sentence-transformers/meta-llama/Llama-3.2-3B-Instruct"
DATA_DIR = "../data/processed/"

In [4]:
file_path = [
    DATA_DIR + "documents.parquet"
]

In [5]:
con = duckdb.connect()
df = con.execute(f"SELECT * FROM '{file_path[0]}'").fetchdf()
df.head()

,title,text,rating,product_title,combined
0,They make my skin feel as smooth as glass,I bought these at Sephora on a whim since I ha...,5.0,OLEHENRIKSEN Grease Relief Cleansing Cloths Pa...,They make my skin feel as smooth as glass I bo...
1,Five Stars,"Nice product. Fits snug, just what I need for ...",5.0,Loritta 5 PCS Women's Headbands Elastic Boho P...,"Five Stars Nice product. Fits snug, just what ..."
2,Oro Gold Body Butter,This Oro Gold stuff is amazing. My husband us...,5.0,"Oro Gold, 24K Golden Body Butter, Devotion, 28...",Oro Gold Body Butter This Oro Gold stuff is am...
3,Three Stars,"Not the easiest to put on.... but, they'll wor...",3.0,AGPtEK Handmade Natural Fashion Long False Eye...,"Three Stars Not the easiest to put on.... but,..."
4,Three Stars,Not a bad product but I it has suffices it FS ...,3.0,OYA Beauty Leave in Conditioner -Size 8.5 oz,Three Stars Not a bad product but I it has suf...


In [6]:
documents = []
for path in file_path:
    loader = DataFrameLoader(df, page_content_column="combined")
    docs = loader.load()

    # Add source metadata to each document
    for doc in docs:
        doc.metadata["source"] = path
    documents.extend(docs)

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

split_docs = text_splitter.split_documents(documents)

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace

In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)